In [ ]:
import collections, io, os, json, subprocess
from pathlib import Path
from pprint import pprint
import pandas as pd
import af3io, pooled_ppi

from procompa import get_data_dir

In [ ]:
# cd get_data_dir() / "26.01_batch-infer/pooled-ppi-yeast/data/predictions-combfold/run_on_pdbs/P00635_P24031_P35842_P38693/input/pdbs"
#!apptainer exec --bind $(pwd) /.../alphafold3/images/alphacutter_c1b8a96.sif /usr/local/bin/_entrypoint.sh \
#    python /app/AlphaCutter_v111.py --loop_min 15 --helix_min 20 --fragment_min 5 --domain_min 0 --pLDDT_min 0 --local_contact_range 5 --single_out --domain_out

In [ ]:
# Examples from Michaelis, Fig 5d - https://www.nature.com/articles/s41586-023-06739-5/figures/5
#proteins = {'P24720', 'P08593', 'P10834'} # 'Mne1', 'Mss18', 'Pet54' - structures missing
#proteins = {'P38013', 'P34760', 'Q04120',} # 'Ahp1', 'Tsa1', 'Tsa2' - reproduces
proteins = {'P24031', 'P00635', 'P35842', 'P38693'} # 'Pho3', 'Pho5', 'Pho11', 'Pho12' - fails, everything aligned into the same blob
complex_name = '_'.join(sorted(proteins))
complex_name

In [ ]:
# Write subunits.json
js = collections.OrderedDict([])
sequences = pd.read_parquet(get_data_dir() / "25.12_pooled-ppi-yeast/data-26.03/proteins.parquet")
for protein, chain_name in zip(proteins, af3io.input.enumerate_chains()):
    #print(protein)
    js[protein] = {
        'name': protein,
        'chain_names': [chain_name],
        'start_res': 1,
        'sequence': sequences.query('uniprot_id == @protein')['seq'].squeeze(),
    }
subunits_path = f'complex-assembly/{complex_name}_input/subunits.json'
Path(subunits_path).parent.mkdir(parents=True, exist_ok=True)
with open(subunits_path, 'w') as fh:
    fh.write(json.dumps(js, indent=2))
#os.system(f'cat {path}')

In [ ]:
# Find pairs, rank by ipTM
pp = pooled_ppi.PooledPredictionsDb(get_data_dir() / "25.12_pooled-ppi-yeast/data-26.03")
pairs_ = pp.pairs.query('uniprot_id1 in @proteins & uniprot_id2 in @proteins').sort_values('af3_pair').reset_index(drop=True)
#pairs_ = pairs_.sort_values('chain_pair_iptm_mean_corrected', ascending=False).reset_index(drop=True)
pairs_['pml_id'] = pairs_['uniprot_id1'].astype(str) + '_' + pairs_['uniprot_id2'].astype(str)
pairs_

In [ ]:
for i, r in pairs_.iterrows():
    db_id1 = f'{r["name"]}_{r.af3_id1}'
    db_id2 = f'{r["name"]}_{r.af3_id2}'
    #path_pdb = f'pdbs/{r.pml_id}.pdb'
    path_pdb = f'complex-assembly/{complex_name}_input/pdbs/AFM_{r.uniprot_id1}_{r.uniprot_id2}_unrelaxed_rank_1_model_1.pdb'
    print(path_pdb)
    Path(path_pdb).parent.mkdir(parents=True, exist_ok=True)
    pp.save_ids([db_id1, db_id2], path_pdb)

In [ ]:
cmd_ = f'python /app/CombFold/scripts/run_on_pdbs.py complex-assembly/{complex_name}_input/subunits.json complex-assembly/{complex_name}_input/pdbs/ complex-assembly/{complex_name}_output/'
cmd_

In [ ]:
!singularity exec /cluster/project/beltrao/shared/alphafold3/images/combfold_latest.sif sh -c '{cmd_}'

visualize pdb sturcures